# OmniDx Vision — training on Google Colab
**One Vision, Multiple Diseases, Smarter Decisions**

This notebook downloads the Kaggle datasets, trains the disease modules on a free GPU, and saves the models and evaluation results to your Google Drive.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.
2. You need a Kaggle API token (`kaggle.json`). Get it from kaggle.com → Settings → API → *Create New Token*.
3. Every number in your report must come from the `metrics.json` files this notebook produces.

> OmniDx provides AI-assisted screening and clinical decision-support information. It is not a substitute for professional medical diagnosis or treatment.

## 1. Check the GPU

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

## 2. Google Drive (models and results are saved here so they survive the session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/OmniDx'
os.makedirs(DRIVE, exist_ok=True)
os.environ['OMNIDX_MODELS'] = f'{DRIVE}/models'   # trained models go to Drive
print('Saving to', DRIVE)

## 3. Get the OmniDx code
Fill in `GITHUB_URL` if the project is on GitHub. Otherwise leave it empty and upload `omnidx-vision.zip` when asked.

In [ ]:
GITHUB_URL = ''   # e.g. 'https://github.com/ThotaSathwika/omnidx-vision.git'

%cd /content
if GITHUB_URL:
    !rm -rf omnidx-vision && git clone -q {GITHUB_URL} omnidx-vision
else:
    from google.colab import files
    up = files.upload()                       # choose omnidx-vision.zip
    !rm -rf omnidx-vision && unzip -q -o {list(up)[0]}
%cd /content/omnidx-vision
!pip install -q -r requirements.txt kaggle
!ls

## 4. Kaggle API token

In [ ]:
import os, json
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json')
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle user:', json.load(open('/root/.kaggle/kaggle.json'))['username'])

## 5. Download and prepare the datasets
Each dataset is converted to the OmniDx layout: class folders plus `metadata.csv` with patient or lesion IDs.

The Kaggle train/test folders are merged on purpose. OmniDx re-splits the data 70/15/15 by patient, because the original splits can put the same patient in both train and test.

Open each Kaggle page once and accept its terms. Then fill in the licence and version in `configs/modules.yaml`.

In [ ]:
KAGGLE = {
    'pneumonia':   'paultimothymooney/chest-xray-pneumonia',
    'brain_tumor': 'masoudnickparvar/brain-tumor-mri-dataset',
    'skin_cancer': 'kmader/skin-cancer-mnist-ham10000',
}
MODULES = ['pneumonia', 'brain_tumor', 'skin_cancer']   # remove any you don't want to train now

for m in MODULES:
    if os.path.exists(f'datasets/{m}/metadata.csv'):
        print(m, 'already prepared'); continue
    !kaggle datasets download -q -d {KAGGLE[m]} -p /content/raw/{m} --unzip
    !python scripts/prepare_kaggle.py {m} /content/raw/{m} datasets/{m}
    !rm -rf /content/raw/{m}

## 6. Train, validate and test
Each run does the following:
1. Checks for patient leakage and duplicate images.
2. Trains the CNN with early stopping.
3. Tests once on the held-out test split.
4. Runs the Grad-CAM audit.
5. Trains the Random Forest fusion layer (`--fusion`).

Expect roughly 10–40 minutes per module on a T4 GPU. Pre-processed images are cached after the first epoch.

`--drop-duplicates` removes test/validation copies of images that also appear in training. The brain MRI dataset is known to contain repeats.

In [ ]:
EPOCHS = {'pneumonia': 15, 'brain_tumor': 20, 'skin_cancer': 20}

for m in MODULES:
    print(f'\n========== {m} ==========')
    !python -m omnidx.train --module {m} --data datasets/{m} --epochs {EPOCHS[m]} --batch 32 --workers 2 --drop-duplicates --fusion

## 7. Results (straight from metrics.json, ready for your report)

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import Image, display

MODELS = Path(os.environ['OMNIDX_MODELS'])
rows = []
for m in MODULES:
    run = MODELS / m / (MODELS / m / 'ACTIVE').read_text().strip()
    met = json.loads((run / 'metrics.json').read_text())
    fus = json.loads((run / 'fusion_metrics.json').read_text())
    leak = json.loads((run / 'leakage.json').read_text())
    rows.append({
        'module': m, 'version': run.name, 'test images': met['n_test'],
        'accuracy': round(met['accuracy'], 4), 'accuracy 95% CI': [round(x, 3) for x in met['accuracy_95ci']],
        'macro-F1': round(met['macro_f1'], 4),
        'ROC-AUC': round(met.get('roc_auc') or met.get('roc_auc_macro_ovr') or float('nan'), 4),
        'ECE': round(met['ece'], 4),
        'Grad-CAM energy in ROI': (met['xai_audit'] or {}).get('mean_heatmap_energy_in_roi'),
        'RF fusion accuracy': round(fus['ablation']['fusion']['accuracy'], 4),
        'duplicates dropped': leak.get('dropped_from_val_test', 0),
    })
summary = pd.DataFrame(rows)
summary.to_csv(f'{DRIVE}/results_summary.csv', index=False)
summary

In [ ]:
# per-class sensitivity / specificity
for m in MODULES:
    run = MODELS / m / (MODELS / m / 'ACTIVE').read_text().strip()
    met = json.loads((run / 'metrics.json').read_text())
    print(f'\n{m}')
    display(pd.DataFrame(met['per_class']).T.round(4))

In [ ]:
# Random Forest ablation: does fusing image + radiomics + clinical data help?
for m in MODULES:
    run = MODELS / m / (MODELS / m / 'ACTIVE').read_text().strip()
    fus = json.loads((run / 'fusion_metrics.json').read_text())
    print(f'\n{m}  (clinical data available: {fus["clinical_features_available"]})')
    display(pd.DataFrame(fus['ablation']).T[['accuracy', 'macro_f1']].round(4))

In [ ]:
# plots for the report
for m in MODULES:
    run = MODELS / m / (MODELS / m / 'ACTIVE').read_text().strip()
    print(m)
    for f in ('confusion_matrix.png', 'roc_curve.png', 'fusion_importance.png'):
        display(Image(filename=str(run / f), width=420))

## 8. Grad-CAM gallery (test images the model never saw)

In [ ]:
import matplotlib.pyplot as plt
from omnidx.inference import Predictor
from omnidx.preprocessing import load_image

for m in MODULES:
    run = MODELS / m / (MODELS / m / 'ACTIVE').read_text().strip()
    test = pd.read_csv(run / 'splits.csv').query('split == "test"').groupby('label').head(2)
    p = Predictor(m)
    fig, axes = plt.subplots(len(test), 3, figsize=(9, 3 * len(test)))
    for row, (_, r) in zip(axes, test.iterrows()):
        res, imgs = p.predict(load_image(r.path))
        for ax, k in zip(row, ('processed', 'segmentation', 'heatmap')):
            ax.imshow(imgs[k]); ax.axis('off')
        row[0].set_title(f'true: {r.label}', fontsize=9)
        row[2].set_title(f'model: {res["top_class"]} ({res["confidence"]:.2f})', fontsize=9)
    fig.suptitle(f'{m} — AI-generated screening results', y=1.0)
    fig.tight_layout()
    fig.savefig(f'{DRIVE}/gradcam_{m}.png', dpi=130)
    plt.show()

## 9. Use the trained models in the API
The models are already on Drive in `OmniDx/models/`. Copy that folder into `omnidx-vision/models/` on your laptop and start the API with `uvicorn backend.main:app`. The `/modules` endpoint then shows each deployed version with its real metrics.

In [ ]:
!cd {DRIVE} && zip -qr omnidx_models.zip models results_summary.csv gradcam_*.png
from google.colab import files
files.download(f'{DRIVE}/omnidx_models.zip')

## 10. COPD and liver cirrhosis (when you have a validated image dataset)
No Kaggle slug is filled in for these two modules. Choose an image dataset, check its licence and labels, and arrange it like this:

```
datasets/copd/emphysema/*.png    datasets/copd/normal/*.png
datasets/liver_cirrhosis/cirrhosis/*.png    datasets/liver_cirrhosis/normal/*.png
```

You can add an optional `metadata.csv` (`filename,patient_id` plus clinical columns such as `age`, `bilirubin`, `albumin`, `ast`, `alt`). Then run the training cell with `MODULES = ['copd']`.

For COPD, state the dataset label exactly. An emphysema dataset does not cover every form of COPD.

In [ ]:
# example once the folders exist:
# !python -m omnidx.train --module copd --data datasets/copd --epochs 20 --drop-duplicates --fusion